# Data Exploration and Cleaning

**Database**: organized collection data stored electronically

## DuckDB
- **DuckDB** is an analytical database designed for fast data analysis
- You can think of it as SQLite, but optimized for analytical workloads (OLAP).

## Why databases beat spreadsheets
### Why would a spreadsheet be problematic?
Imagine putting everything into one Excel spreadsheet.

You might have something like this:
| Building ID | District | Material | Damage | Occupants |
| --- | --- | --- | --- | --- |
| 101 | Kathmandu | Brick | Severe | 5 |
| 102 | Lalitpur | Concrete | Moderate | 3 |
| 103 | Kathmandu | Brick | Minor | 4 |
| 104 | Bhaktapur | Wood | Severe | 6 |


- You can separate data into different sheets.
- You can use IDs to connect information manually.
- However, maintaining relationships and enforcing data integrity constraints is more difficult.

1. Large datasets : Excel has a maximum of 1,048,576 rows per worksheet.
2. Data duplication: Data duplication

    Suppose 10,000 buildings are located in Kathmandu.

        You might repeat the district name 10,000 times.

            If the district name changes or is entered inconsistently, you could end up with:
            - Kathmandu
            - kathmandu
            - Katmandu
            These could be interpreted as three different districts during analysis.

3. Data inconsistency

    Imagine the same building appears in two different spreadsheets.

    One spreadsheet says its building material is Brick, while another says Concrete.

    Which one is correct?

    Updating one spreadsheet doesn't automatically update the other.

    This creates inconsistent information.

### How databases solve these problems: Normalization
Normalization is the process of organizing data into related tables to reduce unnecessary duplication and maintain data consistency.

Instead of storing everything in one large table, we separate the information.

For example:

### Table 1: Buildings

| building_id | material | floors |
| --- | --- | --- |
| 101 | Brick | 3 |
| 102 | Concrete | 5 |
| 103 | Brick | 2 |

---

### Table 2: Districts

| district_id | district_name |
| --- | --- |
| 1 | Kathmandu |
| 2 | Lalitpur |
| 3 | Bhaktapur |

---

### Table 3: Building locations

| building_id | district_id |
| --- | --- |
| 101 | 1 |
| 102 | 2 |
| 103 | 1 |

---

### Table 4: Damage assessments

| building_id | damage_level |
| --- | --- |
| 101 | Severe |
| 102 | Moderate |
| 103 | Minor |

Notice that we don't repeatedly store the district name in the building location table. We store its ID instead.

#### Benefits of normalization:

- No redundancy: Redundancy means storing the same information repeatedly when it doesn't need to be repeated. 
  
    | Order ID | Customer name | Address | Product |
    | --- | --- | --- | --- |
    | 101 | John | Nairobi | Laptop |
    | 102 | John | Nairobi | Phone |
    | 103 | Mary | Mombasa | Tablet |
    | 104 | John | Nairobi | Mouse |

    Notice that John's address is repeated three times.

    If John changes his address, you have to update three rows.

    With normalization, you separate customers and orders.

- Consistency: Update one row, every query instantly reflects the change across the entire database
  
   Consistency means that the same information does not have conflicting values in different places.

    Imagine John moves from Nairobi to Kisumu.

    In the unnormalized table, you might update only two of his three order records.
- Integrity: 
-   Integrity means ensuring that the data follows rules and relationships remain valid.
    A database can enforce a foreign key constraint to ensure that:
    You cannot create an order for a customer who doesn't exist.
    You cannot delete a customer who still has orders,
    Constraints prevent orphaned orders() : orphaned record is like a order of customer id 5 but customer id 5 does not exist
- Clarity: "How many customers named John?" is unambiguous — one row per customer

##### How databases ensure data integrity(Data Integrity Constrains)
1. PRIMARY KEY: Uniquely identifies each record (row) in a table. It cannot contain NULL or duplicate values.
2. FOREIGN KEY: References a primary key (or another eligible unique key) in another table to maintain relationships between tables.
3. NOT NULL: Ensures that a column cannot contain NULL values.
4. CHECK: Ensures that values meet a specified condition.
   - Example: age >= 0
5. UNIQUE: Ensures that values in a column (or combination of columns) are not duplicated.
6. DEFAULT: Automatically assigns a predefined value when no value is provided for a column.


## Basic database terminology
1. **Schema**:A blueprint or structure of a database. It defines which tables exist, what data each table stores, the columns, data types and relationships between tables.
2. **Table**: Organizes data into rows and columns.
3. **Query**: An SQL command used to retrieve, insert, update or delete data in a database.
4. **Index**: A data structure that helps a database locate records faster without scanning the entire table.
5. **Transaction**: Group of operations that succeeds or fail together

The bridge table pattern: when two tables have a many-to-many relationship, databases use a third table — a bridge or junction table — that holds both foreign keys

### Entity-Relationship (ER) Diagrams

**ERD (Entity Relationship Diagram)**: A visual representation of a database that shows:
1. Entities: The tables in the database.
2. Attributes: The columns or fields in each table.
3. Relationships: How the tables are connected.
4. Keys: Primary keys (PK) and foreign keys (FK) used to identify records and establish relationships.

## SQL Execution Order
The mental model: think of SQL execution as a pipeline — you open the table, filter it, group it, filter groups, then decide what to show. The output step (SELECT) always comes after all the filtering steps.

## Why DuckDB Wins at Analytics
1. Columnar storage
   This is one of DuckDB's biggest advantages.
   - Row-based storage: Stores data row by row.
   - Columnar storage: Stores data column by column.
   - Imagine a table with 1 million hospital records and 10 columns. You want to calculate the average length of hospital stay.
    ```
    SELECT AVG(length_of_stay)
    FROM hospital_records;
    ```

    A row-oriented database may need to read through rows containing all 10 columns.

    DuckDB's columnar storage allows it to focus on the length_of_stay column, avoiding unnecessary column reads.
    ![duckdb_sql](../images/Screenshot%202026-11-03%20at%2008.45.10.png)
2. Fast analytical queries
    DuckDB uses vectorized query execution.

    Instead of processing one row at a time, it processes batches of values together.
3. Works directly with data files
   
    DuckDB can query CSV, Parquet and other supported file formats without first importing everything into a traditional database.
    No server required
4. DuckDB is an embedded database.

    This means you don't need to set up and manage a separate database server before using it.
5. Integrates with Python
    DuckDB works well with Python libraries such as Pandas and Polars.

    You can query data using SQL and continue analysing it using Python.

In [22]:
import pandas as pd
import sqlite3
import duckdb
import numpy as np
import time


In [23]:
# connect to the database
conn=sqlite3.connect("../data/nepal.sqlite")

In [24]:
# function to execute the query
def execute_query(query,connection=conn):
    return pd.read_sql_query(query,connection)


In [25]:
# list all tables in the database
query=""" 
    SELECT name
    FROM sqlite_master
    WHERE type='table'
"""
execute_query(query)

,name
0,building_structure
1,building_damage
2,household_demographics
3,id_map


In [26]:
q1=""" 
    SELECT *
    FROM household_demographics

"""
execute_query(q1)

,household_id,gender_household_head,age_household_head,caste_household,education_level_household_head,income_level_household,size_household,is_bank_account_present_in_household
0,101,Male,31.0,Rai,Illiterate,Rs. 10 thousand,3.0,0.0
1,201,Female,62.0,Rai,Illiterate,Rs. 10 thousand,6.0,0.0
2,301,Male,51.0,Gharti/Bhujel,Illiterate,Rs. 10 thousand,13.0,0.0
3,401,Male,48.0,Gharti/Bhujel,Illiterate,Rs. 10 thousand,5.0,0.0
4,501,Male,70.0,Gharti/Bhujel,Illiterate,Rs. 10 thousand,8.0,0.0
...,...,...,...,...,...,...,...,...
249927,23483101,Female,48.0,Gurung,Illiterate,Rs. 10 thousand,5.0,0.0
249928,23483201,Male,29.0,Gurung,Class 10,Rs. 10 thousand,2.0,0.0
249929,23483301,Male,24.0,Gurung,Class 5,Rs. 10 thousand,1.0,0.0
249930,23483401,Female,65.0,Ghale,Illiterate,Rs. 10 thousand,1.0,0.0


In [27]:
q2=""" 
    SELECT *
    FROM id_map 
    """
execute_query(q2)

,household_id,building_id,vdcmun_id,district_id
0,5601,56,7,1
1,6301,63,7,1
2,9701,97,7,1
3,9901,99,7,1
4,11501,115,7,1
...,...,...,...,...
249927,23477401,234774,32,4
249928,23480801,234808,32,4
249929,23482101,234821,32,4
249930,23482801,234828,32,4


In [28]:
# caste distribution by district
q3=""" 
SELECT i.district_id,h.caste_household,COUNT(*) as count
FROM household_demographics h
JOIN id_map i USING(household_id)
GROUP BY i.district_id,h.caste_household
ORDER BY  i.district_id,COUNT desc


"""
df_caste=execute_query(q3)
df_caste



,district_id,caste_household,count
0,1,Chhetree,8943
1,1,Brahman-Hill,4123
2,1,Rai,4119
3,1,Tamang,3796
4,1,Magar,3752
...,...,...,...
214,4,Kalwar,1
215,4,Kalar,1
216,4,Hyolmo,1
217,4,Dhimal,1


In [29]:
# Top 3 caste per district
for district in [1,2,3,4]:
    print(f"\nDistrict {district}")
    top_castes=df_caste[df_caste["district_id"]== district].head(3)
    for _,row in top_castes.iterrows():
        print(f"{row['caste_household']}:{row['count']:,} households")
   


District 1
Chhetree:8,943 households
Brahman-Hill:4,123 households
Rai:4,119 households

District 2
Chhetree:15,312 households
Tamang:11,927 households
Newar:7,451 households

District 3
Tamang:28,402 households
Brahman-Hill:19,183 households
Chhetree:10,867 households

District 4
Gurung:15,119 households
Brahman-Hill:13,043 households
Chhetree:8,766 households


In [30]:
top_castes

,district_id,caste_household,count
156,4,Gurung,15119
157,4,Brahman-Hill,13043
158,4,Chhetree,8766


In [31]:
pd.set_option('display.max_columns',None)

In [34]:
q=""" 
    SELECT *
    FROM building_structure
    LIMIT(5)
"""
df_sqlite_head=execute_query(q)
df_sqlite_head

,building_id,count_floors_pre_eq,count_floors_post_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,height_ft_post_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,condition_post_eq,superstructure
0,1,1,1,9,288,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Used in risk,"Stone, mud mortar"
1,2,1,1,15,364,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Repaired and used,"Stone, mud mortar"
2,3,1,1,20,384,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Repaired and used,"Stone, mud mortar"
3,4,1,1,20,312,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Repaired and used,"Stone, mud mortar"
4,5,1,1,30,308,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Repaired and used,"Stone, mud mortar"


In [35]:
q=""" 
    SELECT building_id,age_building
    FROM building_structure
    LIMIT(5)
"""
df_sqlite_columns=execute_query(q)
df_sqlite_columns

,building_id,age_building
0,1,9
1,2,15
2,3,20
3,4,20
4,5,30


In [39]:
q=""" 
SELECT COUNT(building_id) as count
FROM building_structure
"""

total_buildings=execute_query(q)
total_buildings.loc[0,'count']

np.int64(234835)

In [42]:
q=""" 
    SELECT DISTINCT foundation_type
    FROM building_structure
"""
df_foundation_types_sqlite=execute_query(q)
df_foundation_types_sqlite

,foundation_type
0,Other
1,Mud mortar-Stone/Brick
2,Cement-Stone/Brick
3,Bamboo/Timber
4,RC


In [45]:
q=""" 
    SELECT *
    FROM id_map
    WHERE district_id=4
    LIMIT 5
"""
df_gorkha_sqlite=execute_query(q)
df_gorkha_sqlite

,household_id,building_id,vdcmun_id,district_id
0,16400201,164002,38,4
1,16408101,164081,38,4
2,16408901,164089,38,4
3,16409801,164098,38,4
4,16410301,164103,38,4


# Working with DuckDB

In [48]:
df=duckdb.sql(""" 
              SELECT *
              FROM read_csv_auto('../data/building_structure.csv')
              LIMIT 5
              """).fetchdf()
df

,building_id,count_floors_pre_eq,count_floors_post_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,height_ft_post_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,condition_post_eq,superstructure
0,1,1,1,9,288,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Used in risk,"Stone, mud mortar"
1,2,1,1,15,364,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Repaired and used,"Stone, mud mortar"
2,3,1,1,20,384,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Repaired and used,"Stone, mud mortar"
3,4,1,1,20,312,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Repaired and used,"Stone, mud mortar"
4,5,1,1,30,308,9,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Damaged-Repaired and used,"Stone, mud mortar"


In [50]:
q=""" 
    SELECT COUNT(*) as count
    FROM read_csv_auto('../data/id_map.csv')
    WHERE district_id =4
"""
gorkha_count_duckdb =duckdb.sql(q).fetchdf()
gorkha_count_duckdb.loc[0,'count']


np.int64(75883)

In [55]:
q=""" 
SELECT * EXCLUDE (building_id)
FROM read_csv_auto('../data/id_map.csv')
WHERE district_id = 4;
   
   
"""
duckdb.sql(q).fetchdf()

,household_id,vdcmun_id,district_id
0,16400201,38,4
1,16408101,38,4
2,16408901,38,4
3,16409801,38,4
4,16410301,38,4
...,...,...,...
75878,23477401,32,4
75879,23480801,32,4
75880,23482101,32,4
75881,23482801,32,4


In [56]:
%%writefile duckdb_wrangle.py
"""
duckdb_wrangle.py

Data wrangling module for Project 4: Nepal Earthquake Damage Prediction.
This module provides functions to query CSV data using DuckDB for machine learning classification tasks.

Usage:
    from duckdb_wrangle import wrangle_nepal_data

    # All data (default)
    df = wrangle_nepal_data("./data")

    # Specific district (e.g., Gorkha = district_id 4)
    df_gorkha = wrangle_nepal_data("./data", district_id=4)
"""

import duckdb
import pandas as pd


def wrangle_nepal_data(csv_path="./data", district_id=None):
    """
    Retrieve and prepare Nepal earthquake data for binary classification.

    This function reads CSV files using DuckDB and joins building structure
    with building damage data. The damage_grade column is kept as-is for use
    in creating the target variable in Lesson 2.

    Parameters:
    -----------
    csv_path : str
        Path to the data folder (default: "./data")
    district_id : int, optional
        Filter by specific district (e.g., 4 for Gorkha).
        If None, returns all districts.

    Returns:
    --------
    pd.DataFrame
        DataFrame with features and damage_grade column
    """
    # Build file paths
    structure_path = f"{csv_path}/building_structure.csv"
    damage_path = f"{csv_path}/building_damage.csv"

    # Build WHERE clause
    where_clause = "WHERE b.damage_grade IS NOT NULL"
    if district_id is not None:
        where_clause += f" AND i.district_id = {district_id}"

    # Always join with id_map to get district_id
    query = f"""
        SELECT DISTINCT
            s.building_id,
            i.district_id,
            s.age_building,
            s.plinth_area_sq_ft,
            s.height_ft_pre_eq,
            s.foundation_type,
            s.ground_floor_type,
            b.damage_grade
        FROM read_csv_auto('{structure_path}') s
        JOIN read_csv_auto('{damage_path}') b ON s.building_id = b.building_id
        JOIN read_csv_auto('./data/id_map.csv') i ON s.building_id = i.building_id
        {where_clause}
    """

    return (
        duckdb.sql(query)
        .df()
        .set_index("building_id")
        .assign(
            severe_damage=lambda x: (
                x["damage_grade"]
                .str.contains("Grade 4|Grade 5")
                .fillna(False)
                .astype(int)
            )
        )
        .drop(columns=["damage_grade"])

Writing duckdb_wrangle.py
